In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [15]:
user_ratings = pd.read_csv('data/AnimeList/UserAnimeList.csv', nrows=500000)

In [16]:
# Keep only ratings > 0 (0 means unrated in this dataset)
user_ratings = user_ratings[user_ratings['my_score'] > 0].copy()

# Encode usernames and anime_ids to range [0, n_unique-1]
user_enc = LabelEncoder()
user_ratings['user'] = user_enc.fit_transform(user_ratings['username'])
n_users = user_ratings['user'].nunique()

item_enc = LabelEncoder()
user_ratings['anime'] = item_enc.fit_transform(user_ratings['anime_id'])
n_anime = user_ratings['anime'].nunique()

print(f"Number of Users: {n_users}")
print(f"Number of Anime: {n_anime}")

# Normalize ratings to range [0, 1] for easier training
# Ratings are 1-10, so we can map them to 0-1
user_ratings['rating'] = user_ratings['my_score'].values.astype(np.float32) / 10.0

X = user_ratings[['user', 'anime']].values
y = user_ratings['rating'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Number of Users: 1449
Number of Anime: 10594


In [17]:
# Cell 4: Create PyTorch Dataset
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

class AnimeDataset(Dataset):
    def __init__(self, X, y):
        self.users = torch.tensor(X[:, 0], dtype=torch.long)
        self.anime = torch.tensor(X[:, 1], dtype=torch.long)
        self.ratings = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.anime[idx], self.ratings[idx]

train_ds = AnimeDataset(X_train, y_train)
test_ds = AnimeDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)


In [18]:
# Cell 5: Neural Collaborative Filtering Model
class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_size=50):
        super(NCF, self).__init__()
        # User & Item Embeddings
        self.user_emb = nn.Embedding(n_users, emb_size)
        self.item_emb = nn.Embedding(n_items, emb_size)
        
        # Dense Layers to combine embeddings
        self.fc_layers = nn.Sequential(
            nn.Linear(emb_size * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Output a single score prediction
        )
        
    def forward(self, user, item):
        u = self.user_emb(user)
        i = self.item_emb(item)
        x = torch.cat([u, i], dim=1) # Concatenate user and item embeddings
        return self.fc_layers(x).squeeze()

model = NCF(n_users, n_anime)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [19]:
# Cell 6: Training Loop
epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for u, i, r in train_loader:
        u, i, r = u.to(device), i.to(device), r.to(device)
        optimizer.zero_grad()
        preds = model(u, i)
        loss = criterion(preds, r)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for u, i, r in test_loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            preds = model(u, i)
            val_loss += criterion(preds, r).item()
            
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {total_loss/len(train_loader):.4f} - Val Loss: {val_loss/len(test_loader):.4f}")


Epoch 1/5 - Train Loss: 0.0252 - Val Loss: 0.0201
Epoch 2/5 - Train Loss: 0.0185 - Val Loss: 0.0182
Epoch 3/5 - Train Loss: 0.0172 - Val Loss: 0.0176
Epoch 4/5 - Train Loss: 0.0164 - Val Loss: 0.0174
Epoch 5/5 - Train Loss: 0.0159 - Val Loss: 0.0174
